In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import os

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


TUPLES_PATH = r"quintuplet_tuples/quintuplet_tuples.npy"
EMBED_PATH  = "filtered_embeddings/train_embeddings_2M.npy"
IDS_PATH    = "filtered_embeddings/train_ids_2M.npy"

print("Loading mappings and tuples...")

train_ids = np.load(IDS_PATH)
tuples = np.load(TUPLES_PATH) 

id_to_idx = {photo_id: idx for idx, photo_id in enumerate(train_ids)}

# Filter out tuples where any of the 5 IDs are missing from our processed 2.5M subset
valid_tuples = []
for t in tqdm(tuples, desc="Validating Tuples"):
    if all(photo_id in id_to_idx for photo_id in t):
        valid_tuples.append(t)

valid_tuples = np.array(valid_tuples)
print(f"Total valid quintuplet tuples for training: {len(valid_tuples):,}")

Using device: cuda
Loading mappings and tuples...


Validating Tuples: 100%|██████████████████████████████████████████████████████████████████████████████████████| 2034723/2034723 [00:12<00:00, 164157.14it/s]


Total valid quintuplet tuples for training: 1,426,874


In [6]:
class QuintupletDataset(Dataset):
    def __init__(self, tuples, id_to_idx, embed_path):
        self.tuples = tuples
        self.id_to_idx = id_to_idx
        self.embed_path = embed_path
        
        self.embeddings = np.load(self.embed_path, mmap_mode='r')

    def __len__(self):
        return len(self.tuples)

    def __getitem__(self, idx):
    
        tuple_ids = self.tuples[idx]
    
        row_indices = [self.id_to_idx[pid] for pid in tuple_ids]

        embeds = np.array([self.embeddings[r_idx] for r_idx in row_indices])
        
        return torch.tensor(embeds, dtype=torch.float32)

# Hyperparameters
BATCH_SIZE = 512
NUM_WORKERS = 32

train_dataset = QuintupletDataset(valid_tuples, id_to_idx, EMBED_PATH)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)

print(f"DataLoader ready. Steps per epoch: {len(train_loader)}")

DataLoader ready. Steps per epoch: 2787


In [7]:
class VisualDNAEncoder(nn.Module):
    def __init__(self, input_dim=2048, embed_dim=256):
        super(VisualDNAEncoder, self).__init__()
        
        # Self-Attention Mechanism to weigh significant visual features
        self.attention = nn.Sequential(
            nn.Linear(input_dim, input_dim // 2),
            nn.Tanh(),
            nn.Linear(input_dim // 2, input_dim),
            nn.Sigmoid()
        )

        self.projector = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, embed_dim)
        )

    def forward(self, x):
        # x shape: (Batch, 2048)
        
        # 1. Calculate attention weights
        attn_weights = self.attention(x)
        
        # 2. Apply weights to the original features
        attended_features = x * attn_weights
        
        # 3. Project to the final Visual DNA space
        visual_dna = self.projector(attended_features)
        
        # L2 Normalize the embeddings for distance calculations
        visual_dna = nn.functional.normalize(visual_dna, p=2, dim=1)
        
        return visual_dna

model = VisualDNAEncoder().to(device)
print(model)

VisualDNAEncoder(
  (attention): Sequential(
    (0): Linear(in_features=2048, out_features=1024, bias=True)
    (1): Tanh()
    (2): Linear(in_features=1024, out_features=2048, bias=True)
    (3): Sigmoid()
  )
  (projector): Sequential(
    (0): Linear(in_features=2048, out_features=1024, bias=True)
    (1): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=1024, out_features=256, bias=True)
  )
)


In [8]:
class QuintupletLoss(nn.Module):
    def __init__(self, margins=[0.2, 0.2, 0.2]):
        """
        Margins dictate how far apart the hierarchical levels should be pushed.
        m1: Margin between (same user/same POI) and (diff user/same POI)
        m2: Margin between (diff user/same POI) and (same user/diff POI)
        m3: Margin between (same user/diff POI) and (diff user/diff POI)
        """
        super(QuintupletLoss, self).__init__()
        self.m1, self.m2, self.m3 = margins

    def forward(self, embeddings):
        # embeddings shape: (Batch, 5, Embed_Dim)
        # Split into Query and the 4 similarity levels
        Q  = embeddings[:, 0, :]
        L1 = embeddings[:, 1, :] # Same User, Same POI
        L2 = embeddings[:, 2, :] # Diff User, Same POI
        L3 = embeddings[:, 3, :] # Same User, Diff POI
        L4 = embeddings[:, 4, :] # Diff User, Diff POI
        
        # Calculate pairwise Euclidean distances
        pdist = nn.PairwiseDistance(p=2)
        d_Q_L1 = pdist(Q, L1)
        d_Q_L2 = pdist(Q, L2)
        d_Q_L3 = pdist(Q, L3)
        d_Q_L4 = pdist(Q, L4)
        
        # Enforce the hierarchy: d(Q,L1) < d(Q,L2) < d(Q,L3) < d(Q,L4)
        loss1 = torch.relu(d_Q_L1 - d_Q_L2 + self.m1)
        loss2 = torch.relu(d_Q_L2 - d_Q_L3 + self.m2)
        loss3 = torch.relu(d_Q_L3 - d_Q_L4 + self.m3)
        
        total_loss = torch.mean(loss1 + loss2 + loss3)
        return total_loss

criterion = QuintupletLoss(margins=[0.1, 0.1, 0.2])

In [9]:
EPOCHS = 10
LEARNING_RATE = 1e-4

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

print("Starting Training Phase...")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in loop:
        # Batch shape: (Batch, 5, 2048)
        batch = batch.to(device)
        batch_size = batch.shape[0]
        
        optimizer.zero_grad()
        
        # Reshape to push all 5*Batch images through the network at once
        flat_batch = batch.view(-1, 2048)
        
        # Forward pass
        flat_embeddings = model(flat_batch)
        
        # Reshape back to (Batch, 5, Embed_Dim) for the loss function
        embed_dim = flat_embeddings.shape[-1]
        embeddings = flat_embeddings.view(batch_size, 5, embed_dim)
        
        # Compute Loss
        loss = criterion(embeddings)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())
        
    avg_loss = running_loss / len(train_loader)
    scheduler.step(avg_loss)
    print(f"Epoch [{epoch+1}/{EPOCHS}] Average Loss: {avg_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

# Save the trained encoder for Module C
torch.save(model.state_dict(), "VisualDNA_Encoder_Weights.pth")
print("Model weights saved to VisualDNA_Encoder_Weights.pth")

Starting Training Phase...


Epoch 1/10: 100%|███████████████████████████████████████████████████████████████████████████████████████████| 2787/2787 [01:59<00:00, 23.26it/s, loss=0.306]


Epoch [1/10] Average Loss: 0.3104 | LR: 0.000100


Epoch 2/10: 100%|███████████████████████████████████████████████████████████████████████████████████████████| 2787/2787 [01:46<00:00, 26.21it/s, loss=0.299]


Epoch [2/10] Average Loss: 0.3030 | LR: 0.000100


...

Epoch [8/10] Average Loss: 0.2873 | LR: 0.000100


Epoch 9/10: 100%|███████████████████████████████████████████████████████████████████████████████████████████| 2787/2787 [01:46<00:00, 26.14it/s, loss=0.269]


Epoch [9/10] Average Loss: 0.2851 | LR: 0.000100


Epoch 10/10: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 2787/2787 [01:42<00:00, 27.09it/s, loss=0.294]

Epoch [10/10] Average Loss: 0.2830 | LR: 0.000100
Model weights saved to VisualDNA_Encoder_Weights.pth


In [11]:
import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class VisualDNAEncoder(nn.Module):
    def __init__(self, input_dim=2048, embed_dim=256):
        super(VisualDNAEncoder, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(input_dim, input_dim // 2),
            nn.Tanh(),
            nn.Linear(input_dim // 2, input_dim),
            nn.Sigmoid()
        )
        self.projector = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, embed_dim)
        )

    def forward(self, x):
        attn_weights = self.attention(x)
        attended_features = x * attn_weights
        visual_dna = self.projector(attended_features)
        visual_dna = nn.functional.normalize(visual_dna, p=2, dim=1)
        return visual_dna

# Initialize and load the trained weights
model = VisualDNAEncoder().to(device)
model.load_state_dict(torch.load("VisualDNA_Encoder_Weights.pth", map_location=device))
model.eval() # Crucial: set to evaluation mode (disables dropout, fixes batchnorm)
print("Model loaded successfully and set to evaluation mode.")

Using device: cuda
Model loaded successfully and set to evaluation mode.


In [14]:
import numpy as np
import torch
from tqdm import tqdm

INPUT_EMBED_PATH = "filtered_embeddings/train_embeddings_2M.npy"
OUTPUT_DNA_PATH  = "visual_dna_256d.npy"

input_embeds = np.load(INPUT_EMBED_PATH, mmap_mode='r')
total_images = input_embeds.shape[0]

output_dna = np.lib.format.open_memmap(
    OUTPUT_DNA_PATH, mode='w+', dtype='float32', shape=(total_images, 256)
)

BATCH_SIZE = 4096 

with torch.no_grad(): 
    for i in tqdm(range(0, total_images, BATCH_SIZE), desc="Extracting Visual DNA"):
        batch_end = min(i + BATCH_SIZE, total_images)
        batch_data = input_embeds[i:batch_end]
        
        inputs = torch.tensor(batch_data, dtype=torch.float32).to(device)
        outputs = model(inputs)
        
        output_dna[i:batch_end] = np.array(outputs.cpu().tolist(), dtype=np.float32)

output_dna.flush()
del output_dna 

print(f"\nSuccess! All 256-d Visual DNA vectors saved to {OUTPUT_DNA_PATH}")

Extracting Visual DNA: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 697/697 [1:22:56<00:00,  7.14s/it]



Success! All 256-d Visual DNA vectors saved to visual_dna_256d.npy
